# 07 — Microstructure Recorder (Equities via Alpaca Quotes)

Records real-time equities quote snapshots from Alpaca and stores them in a
LOB-compatible schema for optional real-micro experiments.

Why this shape: current training/live defaults use proxy micro features (`use_proxy=True`),
but `compute_micro_features(..., use_proxy=False)` expects top-of-book style columns.
This notebook writes quote-derived snapshots in that compatible format.

**Implementation note:** Colab uses REST polling (`StockLatestQuoteRequest`) once per interval
for stability.

Captures L1 bid/ask from quotes and writes synthetic L2/L3 depth as zeros for schema compatibility.

**Output schema (per row):**
```
timestamp | bid_price_1 | bid_size_1 | bid_price_2 | ... | ask_size_3
```

**Output:** `/content/drive/MyDrive/algo_trader/data/lob/{SYMBOL}_lob_{date}.parquet`

**Duration:** Set `RECORD_MINUTES` below (default 180 = 3 hours).

Runs continuously until interrupted or `RECORD_MINUTES` elapsed.
Re-run to accumulate optional real-micro datasets per symbol.

In [ ]:
!pip install -q alpaca-py pyarrow pandas tqdm

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
LOB_DIR = '/content/drive/MyDrive/algo_trader/data/lob'
os.makedirs(LOB_DIR, exist_ok=True)
print(f'LOB output directory: {LOB_DIR}')

In [ ]:
# Load API credentials from Colab Secrets (never hard-code)
ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError('Add ALPACA_API_KEY and ALPACA_SECRET_KEY to Colab Secrets first.')
print('Credentials loaded ✓')

In [ ]:
# -- Configuration --
try:
    from tickers import SP100_TICKERS
    SYMBOL = SP100_TICKERS[0]
except Exception:
    SYMBOL = 'AAPL'

RECORD_MINUTES = 180          # Recording duration (180 = 3 hours)
SNAPSHOT_INTERVAL = 60         # Seconds between snapshots (1 per minute)
TOP_LEVELS = 3                 # Kept at 3 for compatibility with utils real-micro path

from datetime import datetime, timezone
DATE_STR = datetime.now(timezone.utc).strftime('%Y%m%d')
SAFE_NAME = SYMBOL.replace('/', '_')
OUT_PATH = f'{LOB_DIR}/{SAFE_NAME}_lob_{DATE_STR}.parquet'

print(f'Symbol:            {SYMBOL}')
print(f'Duration:          {RECORD_MINUTES} minutes')
print(f'Snapshot interval: {SNAPSHOT_INTERVAL} s')
print(f'Top levels:        {TOP_LEVELS} (L1 quote + synthetic L2/L3 zeros)')
print(f'Output file:       {OUT_PATH}')

In [ ]:
# -- Equities quote recording via Alpaca REST polling (Colab-compatible) --
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
from tqdm.notebook import tqdm

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestQuoteRequest

client = StockHistoricalDataClient(
    api_key=ALPACA_API_KEY,
    secret_key=ALPACA_SECRET_KEY,
 )

snapshots = []
end_time = datetime.now(timezone.utc) + timedelta(minutes=RECORD_MINUTES)
n_expected = RECORD_MINUTES * 60 // SNAPSHOT_INTERVAL

print(f'Recording quote-derived micro snapshots for {SYMBOL} for {RECORD_MINUTES} min '
      f'(~{n_expected} snapshots). Interrupt kernel to stop early.')

pbar = tqdm(total=n_expected, desc='Micro snapshots')

try:
    while datetime.now(timezone.utc) < end_time:
        t_start = time.monotonic()
        ts_utc = datetime.now(timezone.utc)

        try:
            req = StockLatestQuoteRequest(symbol_or_symbols=SYMBOL)
            resp = client.get_stock_latest_quote(req)
            quote = resp[SYMBOL]

            row = {'timestamp': ts_utc}

            # L1 from quote
            row['bid_price_1'] = float(quote.bid_price)
            row['bid_size_1'] = float(quote.bid_size)
            row['ask_price_1'] = float(quote.ask_price)
            row['ask_size_1'] = float(quote.ask_size)

            # Synthetic L2/L3 zeros for schema compatibility
            for i in range(2, TOP_LEVELS + 1):
                row[f'bid_price_{i}'] = row['bid_price_1']
                row[f'bid_size_{i}'] = 0.0
                row[f'ask_price_{i}'] = row['ask_price_1']
                row[f'ask_size_{i}'] = 0.0

            snapshots.append(row)
            pbar.update(1)

        except Exception as e:
            print(f'\nPoll error at {ts_utc.strftime("%H:%M:%S UTC")}: {e}')

        elapsed = time.monotonic() - t_start
        sleep_for = max(0.0, SNAPSHOT_INTERVAL - elapsed)
        time.sleep(sleep_for)

except KeyboardInterrupt:
    print('\nRecording interrupted by user.')
finally:
    pbar.close()

print(f'\nCaptured {len(snapshots):,} snapshots.')

In [ ]:
# -- Save snapshots to Parquet --
if not snapshots:
    print('WARNING: No snapshots captured - nothing to save.')
else:
    lob_df = pd.DataFrame(snapshots)
    lob_df = lob_df.set_index('timestamp')
    lob_df.index = pd.to_datetime(lob_df.index, utc=True)

    # Validate basic market micro fields
    assert (lob_df['bid_price_1'] > 0).all(), 'Invalid bid prices'
    assert (lob_df['ask_price_1'] > 0).all(), 'Invalid ask prices'
    assert (lob_df['ask_price_1'] >= lob_df['bid_price_1']).all(), 'Crossed quote detected'

    lob_df.to_parquet(OUT_PATH, compression='snappy')

    spread = lob_df['ask_price_1'] - lob_df['bid_price_1']
    print('=== SAVE SUMMARY ===')
    print(f'  Rows saved:    {len(lob_df):,}')
    print(f'  Symbol:        {SYMBOL}')
    print(f'  Start:         {lob_df.index.min()}')
    print(f'  End:           {lob_df.index.max()}')
    print(f'  Avg spread:    ${spread.mean():.4f}')
    print(f'  Output file:   {OUT_PATH}')
    print('\nSaved quote-derived micro dataset (LOB-compatible schema).')

In [ ]:
# -- Data quality report --
if 'lob_df' in dir() and len(lob_df) > 0:
    spread = lob_df['ask_price_1'] - lob_df['bid_price_1']
    spread_bps = (spread / lob_df['bid_price_1']) * 10_000

    print('=== MICRO DATA QUALITY REPORT ===')
    print(f'  Symbol:             {SYMBOL}')
    print(f'  Total snapshots:    {len(lob_df):,}')
    print(f'  Spread (USD):       mean=${spread.mean():.4f}  max=${spread.max():.4f}')
    print(f'  Spread (bps):       mean={spread_bps.mean():.2f}  max={spread_bps.max():.2f}')
    print(f'  Bid depth (L1):     mean={lob_df["bid_size_1"].mean():.2f} shares')
    print(f'  Ask depth (L1):     mean={lob_df["ask_size_1"].mean():.2f} shares')

    nan_rows = lob_df.isna().any(axis=1).sum()
    print(f'  Rows with NaN:      {nan_rows}')

    # Flag potentially stale or wide spreads for further review
    if spread_bps.mean() > 20:
        print('  WARNING: Average spread > 20 bps - symbol/session may be illiquid.')
    else:
        print('  OK: Spread profile is plausible for equities quote snapshots.')

    zero_depth = ((lob_df['bid_size_1'] <= 0) | (lob_df['ask_size_1'] <= 0)).sum()
    print(f'  Zero-depth rows:    {zero_depth}')
else:
    print('Run recording and save cells first.')